In [1]:
import pandas as pd
import numpy as np
import re

csv_path = "C:/Users/14794/Desktop/Longterm memory/hurst_CI_field_bootstrap_JP_90.csv" 

df = pd.read_csv(csv_path)

# Extract H-label (H1/H2/H3) from the "series" column
df["H_label"] = df["series"].str.extract(r"^(H\d)\b", expand=False)

# Keep only the 3 series we care about
df = df[df["H_label"].isin(["H1", "H2", "H3"])].copy()

# Desired order and labels for the table
order = ["H1", "H2", "H3"]
pretty = {
    "H1": "H1 (Sₜ)",
    "H2": "H2 (Vₜ)",
    "H3": "H3 (Rₜ)",
}

def make_sum_ci_table(period_df: pd.DataFrame, decimals: int = 4) -> pd.DataFrame:
    """
    Build a 3x3 table where entry (i,j) is the CI interval for H_i + H_j:
        [CI_lo(i)+CI_lo(j), CI_hi(i)+CI_hi(j)]
    Diagonal entries correspond to 2*H_i: [2*CI_lo(i), 2*CI_hi(i)].
    """
    # Ensure we have exactly one row per H1/H2/H3
    period_df = period_df.drop_duplicates(subset=["H_label"]).set_index("H_label").loc[order]
    if period_df.index.isnull().any() or len(period_df) != 3:
        raise ValueError("Each period must contain exactly one row for each of H1, H2, H3.")

    lo = period_df["CI_90_lo"].astype(float).to_numpy()
    hi = period_df["CI_90_hi"].astype(float).to_numpy()

    # Pairwise sums via broadcasting
    lo_sum = lo[:, None] + lo[None, :]
    hi_sum = hi[:, None] + hi[None, :]

    # Format as "[lower, upper]"
    fmt = lambda x: f"{x:.{decimals}f}"
    out = np.empty((3, 3), dtype=object)
    for i in range(3):
        for j in range(3):
            out[i, j] = f"[{fmt(lo_sum[i, j])}, {fmt(hi_sum[i, j])}]"

    table = pd.DataFrame(out, index=[pretty[h] for h in order], columns=[pretty[h] for h in order])
    return table

# Build tables for each period
tables = {}
for period, g in df.groupby("period", sort=False):
    tables[period] = make_sum_ci_table(g, decimals=4)

# Print tables
for period, tbl in tables.items():
    print("\n" + "=" * 80)
    print(period)
    print(tbl.to_string())


Period 1
                  H1 (Sₜ)           H2 (Vₜ)           H3 (Rₜ)
H1 (Sₜ)  [0.8275, 1.1630]  [0.8205, 1.1519]  [0.7638, 1.1028]
H2 (Vₜ)  [0.8205, 1.1519]  [0.8135, 1.1408]  [0.7568, 1.0917]
H3 (Rₜ)  [0.7638, 1.1028]  [0.7568, 1.0917]  [0.7001, 1.0425]

Period 2
                  H1 (Sₜ)           H2 (Vₜ)           H3 (Rₜ)
H1 (Sₜ)  [0.8504, 1.0395]  [0.8649, 1.0596]  [0.8711, 1.0749]
H2 (Vₜ)  [0.8649, 1.0596]  [0.8793, 1.0797]  [0.8856, 1.0950]
H3 (Rₜ)  [0.8711, 1.0749]  [0.8856, 1.0950]  [0.8918, 1.1102]

Period 3
                  H1 (Sₜ)           H2 (Vₜ)           H3 (Rₜ)
H1 (Sₜ)  [0.9039, 1.2880]  [0.8670, 1.2576]  [0.6920, 1.0759]
H2 (Vₜ)  [0.8670, 1.2576]  [0.8301, 1.2272]  [0.6551, 1.0455]
H3 (Rₜ)  [0.6920, 1.0759]  [0.6551, 1.0455]  [0.4801, 0.8638]

Period 4
                  H1 (Sₜ)           H2 (Vₜ)           H3 (Rₜ)
H1 (Sₜ)  [0.8498, 1.1401]  [0.7502, 1.0225]  [0.7375, 1.0299]
H2 (Vₜ)  [0.7502, 1.0225]  [0.6506, 0.9049]  [0.6379, 0.9123]
H3 (Rₜ)  [0.7375, 1.0299]  [0.